# Phase 9 — Temporal Hyperparameter Tuning

## TL;DR

- Twelve random-forest configurations are compared with three expanding temporal folds inside the 1990–2007 training period.
- The strongest configuration uses unrestricted tree depth, one observation per terminal leaf, and 80% of features considered at each split.
- After refitting on 1990–2007, the tuned model achieves validation MAE of about 9.6k hg/ha, RMSE of 18.4k hg/ha, and R² of 0.953 on 2008–2010.
- Validation MAE improves by about 5.9% over the initial random-forest candidate, and all predictions remain non-negative.
- Results are stable across five random seeds, but cassava, potatoes, and sweet potatoes remain the most difficult crops.
- The 2011–2013 test partition remains untouched.


## Context & Methods

Hyperparameters are settings chosen before model fitting. This experiment controls three aspects of random-forest complexity:

- `max_depth`: how deep each tree may grow;
- `min_samples_leaf`: the minimum observations allowed in a terminal leaf; and
- `max_features`: the share of features considered at each split.

The tuning folds move forward through time. Each validation period occurs strictly after its training period. The best configuration is chosen by mean temporal MAE, then refitted on the complete 1990–2007 training partition and assessed once on the 2008–2010 outer validation partition.

### Key assumptions

- MAE is the primary selection metric because it is directly interpretable in yield units and is less dominated by extreme errors than RMSE.
- RMSE, R², crop-level error, year-level error, seed stability, and physical plausibility provide supporting evidence.
- The tuning grid is intentionally small and documented to avoid repeatedly searching until one validation score looks unusually good.
- Predictive associations do not establish that rainfall, pesticides, or another feature causes yield changes.


In [ ]:
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from crop_yield.evaluation import (
    prediction_diagnostics,
    regression_metrics,
)
from crop_yield.models import (
    build_random_forest_pipeline,
    build_tuned_random_forest_pipeline,
)
from crop_yield.preprocessing import split_features_target
from crop_yield.splitting import temporal_train_validation_test_split
from crop_yield.tuning import (
    DEFAULT_TUNING_FOLDS,
    RandomForestConfig,
    evaluate_random_forest_configs,
)

plt.style.use("seaborn-v0_8-whitegrid")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_PATH = REPO_ROOT / "data/processed/crop_yield_modeling.csv"


## Data

### 1. Preserve the outer chronological split


In [ ]:
crop_yield = pd.read_csv(DATA_PATH)
temporal_split = temporal_train_validation_test_split(crop_yield)

development_data = temporal_split.train
X_train, y_train = split_features_target(development_data)
X_validation, y_validation = split_features_target(
    temporal_split.validation
)

print(f"Tuning-development rows: {len(development_data):,}")
print(f"Outer-validation rows: {len(X_validation):,}")
print(f"Untouched test rows: {len(temporal_split.test):,}")
print("Tuning folds:")
for fold in DEFAULT_TUNING_FOLDS:
    print(
        f"- {fold.name}: train through {fold.train_end_year}; "
        f"validate {fold.validation_start_year}–"
        f"{fold.validation_end_year}"
    )


## Results

### 2. Compare the documented tuning grid

The first fold's intended 2002–2003 validation window contains only 2002 because 2003 is absent from the source dataset. No rows are invented to fill that gap.


In [ ]:
tuning_configs = [
    RandomForestConfig(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
    )
    for max_depth, min_samples_leaf, max_features in product(
        [None, 18],
        [1, 2, 4],
        [0.5, 0.8],
    )
]

tuning_results = evaluate_random_forest_configs(
    development_data,
    tuning_configs,
    n_estimators=200,
    random_state=42,
)
print(tuning_results.round(2).to_string(index=False))


In [ ]:
plot_data = tuning_results.copy()
plot_data["configuration"] = plot_data.apply(
    lambda row: (
        f"depth={'None' if pd.isna(row['max_depth']) else int(row['max_depth'])}, "
        f"leaf={int(row['min_samples_leaf'])}, "
        f"features={row['max_features']:.1f}"
    ),
    axis=1,
)

fig, ax = plt.subplots(figsize=(10, 6))
ordered = plot_data.sort_values("mean_temporal_mae", ascending=True)
ax.barh(
    ordered["configuration"],
    ordered["mean_temporal_mae"],
    color="#4C956C",
)
ax.invert_yaxis()
ax.set_title("Random-Forest Mean MAE Across Temporal Tuning Folds")
ax.set_xlabel("Mean absolute error (hg/ha; lower is better)")
ax.set_ylabel("Hyperparameter configuration")
plt.tight_layout()
plt.show()


### 3. Select the strongest temporal configuration


In [ ]:
best_configuration = tuning_results.iloc[0]
print("Selected hyperparameters")
print("- max_depth: None")
print(
    f"- min_samples_leaf: "
    f"{int(best_configuration['min_samples_leaf'])}"
)
print(
    f"- max_features: {best_configuration['max_features']:.1f}"
)
print(
    f"- mean temporal MAE: "
    f"{best_configuration['mean_temporal_mae']:,.2f} hg/ha"
)
print(
    f"- worst temporal-fold MAE: "
    f"{best_configuration['worst_temporal_mae']:,.2f} hg/ha"
)


The unrestricted depth does not prove that unlimited complexity is always desirable. In this documented grid, deeper trees generalized better across the earlier time folds. We still inspect the outer-validation score, crop segments, year trend, and random-seed sensitivity before accepting the configuration.


### 4. Compare the initial and tuned forests on outer validation


In [ ]:
outer_models = {
    "Initial random forest": build_random_forest_pipeline(),
    "Tuned random forest": build_tuned_random_forest_pipeline(),
}

outer_predictions = {}
outer_rows = []
for model_name, model in outer_models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_validation)
    outer_predictions[model_name] = predictions
    outer_rows.append(
        {
            "model": model_name,
            **regression_metrics(y_validation.to_numpy(), predictions),
            **prediction_diagnostics(predictions),
        }
    )

outer_results = (
    pd.DataFrame(outer_rows)
    .set_index("model")
    .sort_values("mae")
)
print(outer_results.round(3).to_string())

mae_improvement = (
    outer_results.loc["Initial random forest", "mae"]
    - outer_results.loc["Tuned random forest", "mae"]
) / outer_results.loc["Initial random forest", "mae"]
print(
    f"Tuning reduces outer-validation MAE by "
    f"{mae_improvement:.1%}."
)


### 5. Check sensitivity to the random seed

A random forest samples observations and features. Repeating the selected configuration with several seeds checks whether the conclusion depends on one fortunate random draw.


In [ ]:
seed_rows = []
for seed in [7, 21, 42, 84, 101]:
    seed_model = build_random_forest_pipeline(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        max_features=0.8,
        random_state=seed,
    )
    seed_model.fit(X_train, y_train)
    seed_predictions = seed_model.predict(X_validation)
    seed_rows.append(
        {
            "seed": seed,
            **regression_metrics(
                y_validation.to_numpy(),
                seed_predictions,
            ),
            **prediction_diagnostics(seed_predictions),
        }
    )

seed_results = pd.DataFrame(seed_rows).set_index("seed")
print(seed_results.round(3).to_string())
print(
    f"Validation MAE across seeds: "
    f"{seed_results['mae'].mean():,.2f} ± "
    f"{seed_results['mae'].std():,.2f} hg/ha"
)


### 6. Inspect tuned-model errors by crop and year


In [ ]:
tuned_predictions = outer_predictions["Tuned random forest"]
validation_errors = temporal_split.validation[
    ["item", "year", "yield_hg_per_ha"]
].copy()
validation_errors["prediction"] = tuned_predictions
validation_errors["absolute_error"] = (
    validation_errors["yield_hg_per_ha"]
    - validation_errors["prediction"]
).abs()

crop_errors = (
    validation_errors.groupby("item")
    .agg(
        observations=("absolute_error", "size"),
        mae=("absolute_error", "mean"),
        median_absolute_error=("absolute_error", "median"),
    )
    .sort_values("mae", ascending=False)
)
year_errors = validation_errors.groupby("year").agg(
    observations=("absolute_error", "size"),
    mae=("absolute_error", "mean"),
    median_absolute_error=("absolute_error", "median"),
)

print("Errors by crop")
print(crop_errors.round(2).to_string())
print("Errors by validation year")
print(year_errors.round(2).to_string())
print(
    f"Predictions within 10,000 hg/ha: "
    f"{validation_errors['absolute_error'].le(10_000).mean():.1%}"
)
print(
    f"Predictions within 20,000 hg/ha: "
    f"{validation_errors['absolute_error'].le(20_000).mean():.1%}"
)


## Checks

### 7. Assert the model-selection safeguards


In [ ]:
assert len(tuning_results) == 12
assert pd.isna(best_configuration["max_depth"])
assert int(best_configuration["min_samples_leaf"]) == 1
assert np.isclose(best_configuration["max_features"], 0.8)
assert outer_results.loc["Tuned random forest", "mae"] < 9_700
assert outer_results.loc["Tuned random forest", "r2"] > 0.95
assert (
    outer_results.loc[
        "Tuned random forest",
        "negative_prediction_count",
    ]
    == 0
)
assert seed_results["mae"].std() < 10
assert crop_errors.index[0] == "Cassava"

test_evaluated = False
assert not test_evaluated
print(
    "All Phase 9 tuning checks passed. "
    f"The {len(temporal_split.test):,}-row test set remains untouched."
)


## Takeaways

1. The selected hyperparameters are `max_depth=None`, `min_samples_leaf=1`, and `max_features=0.8`; 300 trees are used for the refitted candidate.
2. Earlier temporal folds and the 2008–2010 outer validation period agree that this configuration is stronger than the more heavily regularized initial forest.
3. Seed-to-seed MAE variation is only a few hg/ha, so the result is not driven by one fortunate random seed.
4. Overall validation performance is strong, but crop-level reliability is uneven. Cassava, potatoes, and sweet potatoes require particular attention during interpretation and eventual monitoring.
5. Validation error increases from 2008 to 2010, which suggests some temporal drift or increasing forecasting difficulty.
6. The next phase interprets the tuned model and checks whether its learned relationships are agriculturally plausible before any final test evaluation.
